# GenHMM1d — R / Python parity check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamadouyamar/GenHMM1d/blob/master/parity.ipynb)

For each model class shared by the two packages, the flow is the same:

1. **R cell** — simulate a series with the CRAN
   [R GenHMM1d](https://cran.r-project.org/package=GenHMM1d) and estimate it
   with the R `EstHMMGen`;
2. **Python cell** — estimate the **same series** with the Python `EstHMMGen`
   (this repository) and print truth, R estimate, Python estimate, and the
   largest R-Python gap.

Both packages use the same EM with the same deterministic initialization, and
the settings are matched (`eps = 1e-4`, `max_iter = 10000`, minimum 100 EM
iterations), so the estimates should agree to numerical tolerance; residual
differences come only from the Nelder-Mead internals of `stats::optim` vs
`scipy.optimize.minimize`. Models: Gaussian, Poisson, zero-inflated Gaussian,
zero-inflated Poisson. (The autoregressive models and the copulas exist only
in the Python package - outside the parity scope.)


In [ ]:
# ---- setup: Python package + constants --------------------------------------
try:
    import genhmm1d
except ImportError:                       # e.g. on Google Colab
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/mamadouyamar/GenHMM1d.git"], check=True)

import numpy as np
from genhmm1d.hmm import HMM
hmm = HMM()

N        = 2000     # sample size
MAX_ITER = 10000
EPS      = 1e-4     # stopping criterion (R default), passed to both sides
NINIT    = 100      # R hardcodes 100 minimum EM iterations; matched in Python
TOL      = 1e-2     # parity tolerance (optimizer internals differ slightly)
results  = {}

%load_ext rpy2.ipython


In [1]:
%%R
# ---- setup: R package (takes a few minutes on Colab) ------------------------
options(repos = "https://cloud.r-project.org")
if (!requireNamespace("GenHMM1d", quietly = TRUE)) install.packages("GenHMM1d")
suppressMessages(library(GenHMM1d))
cat("R GenHMM1d version:", as.character(packageVersion("GenHMM1d")), "\n")


R GenHMM1d version: 0.2.6 


## Gaussian, two regimes

In [1]:
%%R -i N,MAX_ITER,EPS -o y,th_R,Q_R
set.seed(101)

theta = matrix(c(0, 3,
                 1, 1),
               2, 2)              # regime means 0 and 3, sds 1

Q = matrix(c(0.94, 0.03,
             0.06, 0.97),
           2, 2)

sim = SimHMMGen(theta,
                Q = Q,
                family = "gaussian",
                n = N)

y = as.numeric(sim$SimData)

est = EstHMMGen(sim$SimData,
                ZI = 0,
                reg = 2,
                family = "gaussian",
                max_iter = MAX_ITER,
                eps = EPS)

th_R = est$theta; Q_R = est$Q


In [1]:
out = hmm.EstHMMGen(np.asarray(y).reshape(-1, 1), 2, 'norm',
                    max_iter=MAX_ITER, ninit=NINIT, eps=EPS, ZI=0)
th_P, Q_P = np.asarray(out["theta"], float), np.asarray(out["Q"], float)
th_R, Q_R = np.asarray(th_R, float).reshape(th_P.shape), np.asarray(Q_R, float).reshape(2, 2)

# align regime labels (R and Python may order the regimes differently)
o = np.argsort(th_R[:, 0]); th_R, Q_R = th_R[o], Q_R[np.ix_(o, o)]
o = np.argsort(th_P[:, 0]); th_P, Q_P = th_P[o], Q_P[np.ix_(o, o)]

theta_true = np.array([[0.0, 1.0], [3.0, 1.0]])
d_th, d_Q = np.max(np.abs(th_R - th_P)), np.max(np.abs(Q_R - Q_P))
print("theta_true:\n", theta_true)
print("theta_R   :\n", np.round(th_R, 4))
print("theta_Py  :\n", np.round(th_P, 4))
print("Q_R :\n", np.round(Q_R, 4))
print("Q_Py:\n", np.round(Q_P, 4))
print(f"max |R-Py|: theta {d_th:.2e}  Q {d_Q:.2e}  "
      f"[{'PASS' if max(d_th, d_Q) < TOL else 'FAIL'}]")
results["gaussian"] = (d_th, d_Q)


theta_true:
 [[0. 1.]
 [3. 1.]]
theta_R   :
 [[0.0186 1.0359]
 [2.9772 0.9783]]
theta_Py  :
 [[0.0187 1.0359]
 [2.9773 0.9784]]
Q_R :
 [[0.9172 0.0828]
 [0.0312 0.9688]]
Q_Py:
 [[0.9172 0.0828]
 [0.0312 0.9688]]
max |R-Py|: theta 1.11e-04  Q 5.76e-06  [PASS]


## Poisson, two regimes

In [1]:
%%R -i N,MAX_ITER,EPS -o y,th_R,Q_R
set.seed(102)

theta = matrix(c(2, 9),
               2, 1)              # lambda = 2 and 9

Q = matrix(c(0.94, 0.03,
             0.06, 0.97),
           2, 2)

sim = SimHMMGen(theta,
                Q = Q,
                family = "poisson",
                n = N)

y = as.numeric(sim$SimData)

est = EstHMMGen(sim$SimData,
                ZI = 0,
                reg = 2,
                family = "poisson",
                max_iter = MAX_ITER,
                eps = EPS)

th_R = est$theta; Q_R = est$Q


In [1]:
out = hmm.EstHMMGen(np.asarray(y).reshape(-1, 1), 2, 'poisson',
                    max_iter=MAX_ITER, ninit=NINIT, eps=EPS, ZI=0)
th_P, Q_P = np.asarray(out["theta"], float), np.asarray(out["Q"], float)
th_R, Q_R = np.asarray(th_R, float).reshape(th_P.shape), np.asarray(Q_R, float).reshape(2, 2)

# align regime labels (R and Python may order the regimes differently)
o = np.argsort(th_R[:, 0]); th_R, Q_R = th_R[o], Q_R[np.ix_(o, o)]
o = np.argsort(th_P[:, 0]); th_P, Q_P = th_P[o], Q_P[np.ix_(o, o)]

theta_true = np.array([[2.0], [9.0]])
d_th, d_Q = np.max(np.abs(th_R - th_P)), np.max(np.abs(Q_R - Q_P))
print("theta_true:\n", theta_true)
print("theta_R   :\n", np.round(th_R, 4))
print("theta_Py  :\n", np.round(th_P, 4))
print("Q_R :\n", np.round(Q_R, 4))
print("Q_Py:\n", np.round(Q_P, 4))
print(f"max |R-Py|: theta {d_th:.2e}  Q {d_Q:.2e}  "
      f"[{'PASS' if max(d_th, d_Q) < TOL else 'FAIL'}]")
results["poisson"] = (d_th, d_Q)


theta_true:
 [[2.]
 [9.]]
theta_R   :
 [[1.9766]
 [9.0674]]
theta_Py  :
 [[1.9766]
 [9.0674]]
Q_R :
 [[0.9452 0.0548]
 [0.034  0.966 ]]
Q_Py:
 [[0.9452 0.0548]
 [0.034  0.966 ]]
max |R-Py|: theta 6.86e-05  Q 2.43e-07  [PASS]


## Zero-inflated Gaussian, two regimes

In [1]:
%%R -i N,MAX_ITER,EPS -o y,th_R,Q_R
set.seed(103)

theta = matrix(c(0, 3,
                 0, 1),
               2, 2)              # row 1 = point mass at 0

Q = matrix(c(0.94, 0.03,
             0.06, 0.97),
           2, 2)

sim = SimHMMGen(theta,
                Q = Q,
                ZI = 1,
                family = "gaussian",
                n = N)

y = as.numeric(sim$SimData)

est = EstHMMGen(sim$SimData,
                ZI = 1,
                reg = 2,
                family = "gaussian",
                max_iter = MAX_ITER,
                eps = EPS)

th_R = est$theta; Q_R = est$Q


In [1]:
out = hmm.EstHMMGen(np.asarray(y).reshape(-1, 1), 2, 'norm',
                    max_iter=MAX_ITER, ninit=NINIT, eps=EPS, ZI=1)
th_P, Q_P = np.asarray(out["theta"], float), np.asarray(out["Q"], float)
th_R, Q_R = np.asarray(th_R, float).reshape(th_P.shape), np.asarray(Q_R, float).reshape(2, 2)

# align regime labels (R and Python may order the regimes differently)
o = np.argsort(th_R[:, 0]); th_R, Q_R = th_R[o], Q_R[np.ix_(o, o)]
o = np.argsort(th_P[:, 0]); th_P, Q_P = th_P[o], Q_P[np.ix_(o, o)]

theta_true = np.array([[0.0, 0.0], [3.0, 1.0]])
d_th, d_Q = np.max(np.abs(th_R - th_P)), np.max(np.abs(Q_R - Q_P))
print("theta_true:\n", theta_true)
print("theta_R   :\n", np.round(th_R, 4))
print("theta_Py  :\n", np.round(th_P, 4))
print("Q_R :\n", np.round(Q_R, 4))
print("Q_Py:\n", np.round(Q_P, 4))
print(f"max |R-Py|: theta {d_th:.2e}  Q {d_Q:.2e}  "
      f"[{'PASS' if max(d_th, d_Q) < TOL else 'FAIL'}]")
results["zi-gaussian"] = (d_th, d_Q)


theta_true:
 [[0. 0.]
 [3. 1.]]
theta_R   :
 [[0.     0.    ]
 [3.0251 0.9814]]
theta_Py  :
 [[0.     0.    ]
 [3.0252 0.9815]]
Q_R :
 [[0.9471 0.0529]
 [0.024  0.976 ]]
Q_Py:
 [[0.9471 0.0529]
 [0.024  0.976 ]]
max |R-Py|: theta 1.28e-04  Q 3.33e-16  [PASS]


## Zero-inflated Poisson, two regimes

In [1]:
%%R -i N,MAX_ITER,EPS -o y,th_R,Q_R
set.seed(104)

theta = matrix(c(0, 9),
               2, 1)              # row 1 = point mass at 0

Q = matrix(c(0.94, 0.03,
             0.06, 0.97),
           2, 2)

sim = SimHMMGen(theta,
                Q = Q,
                ZI = 1,
                family = "poisson",
                n = N)

y = as.numeric(sim$SimData)

est = EstHMMGen(sim$SimData,
                ZI = 1,
                reg = 2,
                family = "poisson",
                max_iter = MAX_ITER,
                eps = EPS)

th_R = est$theta; Q_R = est$Q


In [1]:
out = hmm.EstHMMGen(np.asarray(y).reshape(-1, 1), 2, 'poisson',
                    max_iter=MAX_ITER, ninit=NINIT, eps=EPS, ZI=1)
th_P, Q_P = np.asarray(out["theta"], float), np.asarray(out["Q"], float)
th_R, Q_R = np.asarray(th_R, float).reshape(th_P.shape), np.asarray(Q_R, float).reshape(2, 2)

# align regime labels (R and Python may order the regimes differently)
o = np.argsort(th_R[:, 0]); th_R, Q_R = th_R[o], Q_R[np.ix_(o, o)]
o = np.argsort(th_P[:, 0]); th_P, Q_P = th_P[o], Q_P[np.ix_(o, o)]

theta_true = np.array([[0.0], [9.0]])
d_th, d_Q = np.max(np.abs(th_R - th_P)), np.max(np.abs(Q_R - Q_P))
print("theta_true:\n", theta_true)
print("theta_R   :\n", np.round(th_R, 4))
print("theta_Py  :\n", np.round(th_P, 4))
print("Q_R :\n", np.round(Q_R, 4))
print("Q_Py:\n", np.round(Q_P, 4))
print(f"max |R-Py|: theta {d_th:.2e}  Q {d_Q:.2e}  "
      f"[{'PASS' if max(d_th, d_Q) < TOL else 'FAIL'}]")
results["zi-poisson"] = (d_th, d_Q)


theta_true:
 [[0.]
 [9.]]
theta_R   :
 [[0.    ]
 [9.0614]]
theta_Py  :
 [[0.    ]
 [9.0604]]
Q_R :
 [[0.9398 0.0602]
 [0.0315 0.9685]]
Q_Py:
 [[0.94   0.06  ]
 [0.0314 0.9686]]
max |R-Py|: theta 1.00e-03  Q 1.46e-04  [PASS]


## Parity summary

In [1]:
print(f"{'model':<14}{'max|dtheta|':>14}{'max|dQ|':>12}   verdict")
for k, (dt, dq) in results.items():
    print(f"{k:<14}{dt:>14.2e}{dq:>12.2e}   "
          f"{'PASS' if max(dt, dq) < TOL else 'FAIL'}")


model            max|dtheta|     max|dQ|   verdict
gaussian            1.11e-04    5.76e-06   PASS
poisson             6.86e-05    2.43e-07   PASS
zi-gaussian         1.28e-04    3.33e-16   PASS
zi-poisson          1.00e-03    1.46e-04   PASS


### Reading the result

- **PASS everywhere**: on identical data the Python package reproduces the
  CRAN R package to numerical tolerance on every shared model class - the two
  implementations are the same estimator.
- **A FAIL is informative**: with identical data and deterministic
  initialization on both sides, a gap beyond optimizer tolerance points to a
  genuine difference between the code bases and identifies exactly which
  family and parameter to inspect.
